# Setup

In [1]:
# Install the YAML magic
!pip install yamlmagic --quiet
%load_ext yamlmagic

In [2]:
import os

job_name = os.environ.get("JOB_NAME", "sft")
model_name = os.environ.get("MODEL_NAME", "meta-llama/Llama-3.2-1B-Instruct")
# model_name = os.environ.get("MODEL_NAME", "meta-llama/Llama-3.1-8B-Instruct")
resume_from_checkpoint = os.environ.get("RESUME_FROM_CHECKPOINT", "false").lower() in ("true", "1", "yes", "y")
num_workers = int(os.environ.get("NUM_WORKERS", "1"))
# num_gpu_per_worker = int(os.environ.get("NUM_GPU_PER_WORKER", "1"))
num_gpu_per_worker = int(os.environ.get("NUM_GPU_PER_WORKER", "4"))
worker_cpu = int(os.environ.get("WORKER_CPU", "8"))
worker_memory = int(os.environ.get("WORKER_MEMORY", "32"))
batch_size = int(os.environ.get("BATCH_SIZE", "40"))
# batch_size = int(os.environ.get("BATCH_SIZE", "20"))

hf_token = os.environ.get("HF_TOKEN")
openshift_api_token = os.environ.get("OPENSHIFT_API_TOKEN")

openshift_api_server = "https://kubernetes.default.svc"
training_base_image = "quay.io/modh/training:py311-cuda121-torch241"

job_base_dir = "/mnt/shared"
hf_home = f"{job_base_dir}/.cache"
output_dir = f"{job_base_dir}/{model_name}"

local_base_dir = os.path.expanduser("~/shared")
local_hf_home = f"{local_base_dir}/.cache"
local_output_dir = f"{local_base_dir}/{model_name}"
merged_subdir = "merged_model"

# Training Configuration

Edit the following training parameters:

In [3]:
%%yaml parameters

# Model
model_name_or_path: <model_name_or_path>
model_revision: main
torch_dtype: bfloat16
attn_implementation: flash_attention_2    # one of eager (default), sdpa or flash_attention_2
use_liger: false                          # use Liger kernels

# PEFT / LoRA
use_peft: true
lora_r: 16
lora_alpha: 8
lora_dropout: 0.05
lora_target_modules: ["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
lora_modules_to_save: []

# QLoRA (BitsAndBytes)
load_in_4bit: false                       # use 4 bit precision for the base model (only with LoRA)
load_in_8bit: false                       # use 8 bit precision for the base model (only with LoRA)

# Dataset
dataset_name: gsm8k                       # id or path to the dataset
dataset_config: main                      # name of the dataset configuration
dataset_train_split: train                # dataset split to use for training
dataset_test_split: test                  # dataset split to use for evaluation
dataset_text_field: text                  # name of the text field of the dataset
dataset_kwargs:
  add_special_tokens: false               # template with special tokens
  append_concat_token: false              # add additional separator token

# SFT
max_seq_length: 1024                      # max sequence length for model and packing of the dataset
dataset_batch_size: 1000                  # samples to tokenize per batch
packing: false

# Training
num_train_epochs: 10                      # number of training epochs

per_device_train_batch_size: 40           # batch size per device during training
per_device_eval_batch_size: 40            # batch size for evaluation
auto_find_batch_size: false               # find a batch size that fits into memory automatically
eval_strategy: epoch                      # evaluate every epoch

bf16: true                                # use bf16 16-bit (mixed) precision
tf32: false                               # use tf32 precision

learning_rate: 2.0e-4                     # initial learning rate
warmup_steps: 10                          # steps for a linear warmup from 0 to `learning_rate`
lr_scheduler_type: inverse_sqrt           # learning rate scheduler (see transformers.SchedulerType)
# lr_scheduler_type: reduce_lr_on_plateau
# lr_scheduler_kwargs:
  # patience: 1
  # factor: 0.2
# metric_for_best_model: eval_loss

optim: adamw_torch_fused                  # optimizer (see transformers.OptimizerNames)
max_grad_norm: 1.0                        # max gradient norm
seed: 42

gradient_accumulation_steps: 1            # number of steps before performing a backward/update pass
gradient_checkpointing: false             # use gradient checkpointing to save memory
gradient_checkpointing_kwargs:
  use_reentrant: false

# FSDP
fsdp: "full_shard auto_wrap"              # add offload if not enough GPU memory
fsdp_config:
  activation_checkpointing: true
  cpu_ram_efficient_loading: true
  sync_module_states: true
  use_orig_params: true
  limit_all_gathers: true
# fsdp: ""
# fsdp_config: {}

# Checkpointing
save_strategy: epoch                      # save checkpoint every epoch
save_total_limit: 1                       # limit the total amount of checkpoints
resume_from_checkpoint: false             # load the last checkpoint in output_dir and resume from it

# Logging
log_level: warning                        # logging level (see transformers.logging)
logging_strategy: steps
logging_steps: 1                          # log every N steps
report_to:
- tensorboard                             # report metrics to tensorboard

output_dir: <output_dir>

<IPython.core.display.Javascript object>

In [4]:
parameters["model_name_or_path"] = model_name
parameters["output_dir"] = output_dir
parameters["resume_from_checkpoint"] = resume_from_checkpoint

if num_workers * num_gpu_per_worker == 1:
    parameters["fsdp"] = ""

parameters["per_device_train_batch_size"] = batch_size
parameters["per_device_eval_batch_size"] = batch_size

parameters

{'model_name_or_path': 'meta-llama/Llama-3.2-1B-Instruct',
 'model_revision': 'main',
 'torch_dtype': 'bfloat16',
 'attn_implementation': 'flash_attention_2',
 'use_liger': False,
 'use_peft': True,
 'lora_r': 16,
 'lora_alpha': 8,
 'lora_dropout': 0.05,
 'lora_target_modules': ['q_proj',
  'v_proj',
  'k_proj',
  'o_proj',
  'gate_proj',
  'up_proj',
  'down_proj'],
 'lora_modules_to_save': [],
 'load_in_4bit': False,
 'load_in_8bit': False,
 'dataset_name': 'gsm8k',
 'dataset_config': 'main',
 'dataset_train_split': 'train',
 'dataset_test_split': 'test',
 'dataset_text_field': 'text',
 'dataset_kwargs': {'add_special_tokens': False, 'append_concat_token': False},
 'max_seq_length': 1024,
 'dataset_batch_size': 1000,
 'packing': False,
 'num_train_epochs': 10,
 'per_device_train_batch_size': 40,
 'per_device_eval_batch_size': 40,
 'auto_find_batch_size': False,
 'eval_strategy': 'epoch',
 'bf16': True,
 'tf32': False,
 'learning_rate': 0.0002,
 'warmup_steps': 10,
 'lr_scheduler_type

# Training Loop

Review the training function. You can adjust the chat template if needed depending on the model you want to fine-tune:

In [5]:
def main(parameters):
    import random

    from datasets import load_dataset
    from transformers import (
        AutoTokenizer,
        set_seed,
    )

    from trl import (
        ModelConfig,
        ScriptArguments,
        SFTConfig,
        SFTTrainer,
        TrlParser,
        get_peft_config,
        get_quantization_config,
        get_kbit_device_map,
    )

    parser = TrlParser((ScriptArguments, SFTConfig, ModelConfig))
    script_args, training_args, model_args = parser.parse_dict(parameters)

    # Set seed for reproducibility
    set_seed(training_args.seed)

    # Model and tokenizer
    quantization_config = get_quantization_config(model_args)
    model_kwargs = dict(
        revision=model_args.model_revision,
        trust_remote_code=model_args.trust_remote_code,
        attn_implementation=model_args.attn_implementation,
        torch_dtype=model_args.torch_dtype,
        use_cache=False if training_args.gradient_checkpointing or
                           training_args.fsdp_config.get("activation_checkpointing",
                                                         False) else True,
        device_map=get_kbit_device_map() if quantization_config is not None else None,
        quantization_config=quantization_config,
    )
    training_args.model_init_kwargs = model_kwargs
    tokenizer = AutoTokenizer.from_pretrained(
        model_args.model_name_or_path, trust_remote_code=model_args.trust_remote_code, use_fast=True
    )
    if tokenizer.pad_token is None:
        # Models like Llama 3 use a dedicated padding token
        right_pad_id = tokenizer.convert_tokens_to_ids('<|finetune_right_pad_id|>')
        if right_pad_id is not None:
            tokenizer.pad_token = '<|finetune_right_pad_id|>'
        else:
            tokenizer.pad_token = tokenizer.eos_token

    # Chat template
    # You may need to provide your own chat template if the model does not have a default one
    # or if you want to customize it
    # Llama 3 instruct template, make sure to add "lm_head" and "embed_tokens" layers to lora_modules_to_save
    # LLAMA_3_CHAT_TEMPLATE="{% set loop_messages = messages %}{% for message in loop_messages %}{% set content = '<|start_header_id|>' + message['role'] + '<|end_header_id|>\n\n'+ message['content'] | trim + '<|eot_id|>' %}{% if loop.index0 == 0 %}{% set content = bos_token + content %}{% endif %}{{ content }}{% endfor %}{% if add_generation_prompt %}{{ '<|start_header_id|>assistant<|end_header_id|>\n\n' }}{% endif %}"
    # Anthropic/Vicuna like template without the need for special tokens
    # LLAMA_3_CHAT_TEMPLATE = (
    #     "{% for message in messages %}"
    #     "{% if message['role'] == 'system' %}"
    #     "{{ message['content'] }}"
    #     "{% elif message['role'] == 'user' %}"
    #     "{{ '\n\nHuman: ' + message['content'] +  eos_token }}"
    #     "{% elif message['role'] == 'assistant' %}"
    #     "{{ '\n\nAssistant: '  + message['content'] +  eos_token  }}"
    #     "{% endif %}"
    #     "{% endfor %}"
    #     "{% if add_generation_prompt %}"
    #     "{{ '\n\nAssistant: ' }}"
    #     "{% endif %}"
    # )
    # tokenizer.chat_template = LLAMA_3_CHAT_TEMPLATE

    # Datasets
    train_dataset = load_dataset(
        path=script_args.dataset_name,
        name=script_args.dataset_config,
        split=script_args.dataset_train_split,
    )
    test_dataset = None
    if training_args.eval_strategy != "no":
        test_dataset = load_dataset(
            path=script_args.dataset_name,
            name=script_args.dataset_config,
            split=script_args.dataset_test_split,
        )

    # Templatize datasets
    # You may need to adjust the mapping between columns and the chat template
    def template_dataset(sample):
        # return {"text": tokenizer.apply_chat_template(examples["messages"], tokenize=False)}
        messages = [
            {"role": "user", "content": sample['question']},
            {"role": "assistant", "content": sample['answer']},
        ]
        return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

    train_dataset = train_dataset.map(template_dataset, remove_columns=["question", "answer"])
    if training_args.eval_strategy != "no":
        # test_dataset = test_dataset.map(template_dataset, remove_columns=["messages"])
        test_dataset = test_dataset.map(template_dataset, remove_columns=["question", "answer"])

    # Check random samples
    with training_args.main_process_first(
        desc="Log few samples from the training set"
    ):
        for index in random.sample(range(len(train_dataset)), 2):
            print(train_dataset[index]["text"])

    # Training
    trainer = SFTTrainer(
        model=model_args.model_name_or_path,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        peft_config=get_peft_config(model_args),
        processing_class=tokenizer,
    )

    if trainer.accelerator.is_main_process and hasattr(trainer.model, "print_trainable_parameters"):
        trainer.model.print_trainable_parameters()

    checkpoint = None
    if training_args.resume_from_checkpoint is not None:
        checkpoint = training_args.resume_from_checkpoint

    trainer.train(resume_from_checkpoint=checkpoint)

    trainer.save_model(training_args.output_dir)

    with training_args.main_process_first(desc="Training completed"):
        print(f"Training completed, model checkpoint written to {training_args.output_dir}")

# Training Client

Configure the SDK client by providing the authentication token:

In [6]:
# IMPORTANT: Labels and annotations support in create_job() method requires kubeflow-training v1.9.2+. Skip this cell if using RHOAI 2.21 or later.
%pip install -U kubeflow-training --quiet

Note: you may need to restart the kernel to use updated packages.


In [7]:
from kubernetes import client
from kubeflow.training import TrainingClient
from kubeflow.training.models import V1Volume, V1VolumeMount, V1PersistentVolumeClaimVolumeSource, V1EmptyDirVolumeSource

configuration = client.Configuration()
configuration.host = openshift_api_server
configuration.api_key = {"authorization": f"Bearer {openshift_api_token}"}
# Un-comment if your cluster API server uses a self-signed certificate or an un-trusted CA
configuration.verify_ssl = False
api_client = client.ApiClient(configuration)
client = TrainingClient(client_configuration=api_client.configuration)

# Training Job

You're now almost ready to create the training job:
* Fill the `HF_TOKEN` environment variable with your HuggingFace token if you fine-tune a gated model 
* Check the number of worker nodes
* Amend the resources per worker according to the job requirements
* If you use AMD accelerators:
  * Change `nvidia.com/gpu` to `amd.com/gpu` in `resources_per_worker`
  * Change `base_image` to `quay.io/modh/training:py311-rocm62-torch251`
* Update the PVC name to the one you've attached to the workbench if needed

In [8]:
import os
import shutil

if not resume_from_checkpoint:
    print(f"empty local_output_dir: {local_output_dir}")
    if os.path.exists(local_output_dir):
        shutil.rmtree(local_output_dir)

empty local_output_dir: /opt/app-root/src/shared/meta-llama/Llama-3.2-1B-Instruct


In [9]:
try:
    client.delete_job(name=job_name)
except RuntimeError:
    pass

/opt/app-root/lib64/python3.12/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'kubernetes.default.svc'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


In [10]:
client.create_job(
    job_kind="PyTorchJob",
    name=job_name,
    train_func=main,
    num_workers=num_workers,
    num_procs_per_worker=num_gpu_per_worker,
    resources_per_worker={
        "nvidia.com/gpu": num_gpu_per_worker,
        "memory": f"{worker_memory}Gi",
        "cpu": worker_cpu,
    },
    base_image=training_base_image,
    env_vars={
        # HuggingFace
        "HF_HOME": hf_home,
        "HF_TOKEN": hf_token,
        # CUDA / ROCm (HIP)
        "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
        "PYTORCH_HIP_ALLOC_CONF": "expandable_segments:True",
        # NCCL / RCCL
        "NCCL_DEBUG": "INFO",
    },
    # labels={"kueue.x-k8s.io/queue-name": "<LOCAL_QUEUE_NAME>"}, # Optional: Add local queue name and uncomment these lines if using Kueue for resource management
    parameters=parameters,
    volumes=[
        V1Volume(name="shared",
                 persistent_volume_claim=V1PersistentVolumeClaimVolumeSource(claim_name="shared")),
        V1Volume(name="dshm",
                 empty_dir=V1EmptyDirVolumeSource(medium="Memory", size_limit="8Gi"),
        )
    ],
    volume_mounts=[
        V1VolumeMount(name="shared", mount_path="/mnt/shared"),
        V1VolumeMount(name="dshm", mount_path="/dev/shm"),
    ],
)

/opt/app-root/lib64/python3.12/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'kubernetes.default.svc'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


Once the training job is created, you can follow its progress:

In [11]:
import time

job_kind = "PyTorchJob"

# 等待 Job 就绪
while True:
    job_obj = client.get_job(name=job_name, job_kind=job_kind)

    # 通过对象属性访问 status
    job_status = job_obj.status  # 这是一个对象，不是 dict

    # 检查 conditions 是否存在
    conditions = getattr(job_status, "conditions", None)
    if conditions:
        # 找到 type=Running 且 status=True 的 condition
        if any(c.type == "Running" and c.status == "True" for c in conditions):
            break

    time.sleep(5)

# Pod 准备好后，再获取日志
_ = client.get_job_logs(
    name=job_name,
    job_kind=job_kind,
    follow=True,
)

/opt/app-root/lib64/python3.12/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'kubernetes.default.svc'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/opt/app-root/lib64/python3.12/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'kubernetes.default.svc'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/opt/app-root/lib64/python3.12/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'kubernetes.default.svc'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/opt/app-root/lib64/python3.12/

[Pod sft-master-0]: W1101 12:13:36.294000 140634264745792 torch/distributed/run.py:779] 
[Pod sft-master-0]: W1101 12:13:36.294000 140634264745792 torch/distributed/run.py:779] *****************************************
[Pod sft-master-0]: W1101 12:13:36.294000 140634264745792 torch/distributed/run.py:779] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
[Pod sft-master-0]: W1101 12:13:36.294000 140634264745792 torch/distributed/run.py:779] *****************************************
[Pod sft-master-0]: [W1101 12:13:39.146121639 CUDAAllocatorConfig.h:28] Warning: expandable_segments not supported on this platform (function operator())
[Pod sft-master-0]: [W1101 12:13:39.147372384 CUDAAllocatorConfig.h:28] Warning: expandable_segments not supported on this platform (function operator())
[Pod sft-master-0]: [W1101 12:13:39.177495525

# TensorBoard

You can track your job runs and visualize the training metrics with TensorBoard:

In [12]:
import os

nb_prefix = os.environ.get("NB_PREFIX", "")
os.environ["TENSORBOARD_PROXY_URL"] = nb_prefix + "/proxy/6006/"

In [13]:
%load_ext tensorboard

In [14]:
%tensorboard --logdir /opt/app-root/src/shared

# Testing

## Testing the Pre-Trained Model

If you've configured the workbench with a NVIDIA GPU or AMD accelerator, you can run inferences to validate the output generated by the fine-tuned model and compare it to the output of the pre-trained model.

In [15]:
# Install / upgrade dependencies
!pip install --upgrade transformers peft tiktoken --quiet

In [16]:
import torch
import json
import transformers

from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from peft import LoraConfig, PeftModel
from IPython.display import display, Markdown

Check / update the paths to the pre-trained and fine-tuned model checkpoints prior to executing the cells below. 

In [17]:
# 从参数获取模型名
model_name_or_path = parameters["model_name_or_path"]

# Hugging Face Hub 缓存目录
hf_cache_dir = os.path.join(local_hf_home, "hub")

# Hugging Face Hub 会把 model_name_or_path 转成 models--namespace--repo_name 的结构
repo_folder_name = f"models--{model_name_or_path.replace('/', '--')}"
repo_path = os.path.join(hf_cache_dir, repo_folder_name)

# snapshots 文件夹
snapshots_path = os.path.join(repo_path, "snapshots")

# 获取最新的 snapshot（按字母排序）
snapshot_dirs = sorted(os.listdir(snapshots_path))
latest_snapshot = snapshot_dirs[-1]  # 最新 snapshot
pretrained_path = os.path.join(snapshots_path, latest_snapshot)

print("Loading model from:", pretrained_path)

Loading model from: /opt/app-root/src/shared/.cache/hub/models--meta-llama--Llama-3.2-1B-Instruct/snapshots/9213176726f574b556790deb65791e0c5aa438b6


In [18]:
base_model = AutoModelForCausalLM.from_pretrained(
    pretrained_path,
    local_files_only=True,
    torch_dtype=torch.bfloat16,
).to("cuda")

`torch_dtype` is deprecated! Use `dtype` instead!
/opt/app-root/lib64/python3.12/site-packages/torch/cuda/__init__.py:789: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [19]:
# Configure the tokenizer
tokenizer = AutoTokenizer.from_pretrained(pretrained_path)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
if base_model.config.pad_token_id is None:
    base_model.config.pad_token_id = base_model.config.eos_token_id

In [20]:
# Test the pre-trained model
pipeline = transformers.pipeline(
    "text-generation",
    model=base_model,
    tokenizer=tokenizer,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map="auto",
)

messages = [
    {
        "role": "user",
        "content": "Janet's ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?",
    }
]

outputs = pipeline(messages, max_new_tokens=256, temperature = 0.01)

output1 = ""
for turn in outputs:
    for item in turn["generated_text"]:
        output1 += f"# {item['role']}\n\n{item['content']}\n\n"

display(Markdown(output1))

`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda:0


# user

Janet's ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

# assistant

To find out how much Janet makes at the farmers' market, we need to calculate how many eggs she sells and then multiply that by the price per egg.

Janet lays 16 eggs per day. She eats 3 for breakfast, so she has 16 - 3 = 13 eggs left.

She bakes muffins for 4 eggs per day. So, she has 13 - 4 = 9 eggs left.

She sells the remaining 9 eggs at the farmers' market for $2 per egg. So, she makes 9 * $2 = $18 per day.

Janet makes $18 every day at the farmers' market.



## Merging the LoRA adapters

If you've configured the training to use LoRA, then you can merge the fine-tuned LoRA adapters / layers into the pre-trained model.

In [21]:
# Merge the fine-tuned adapters into the base model 
finetuned_path = output_dir.replace(job_base_dir, local_base_dir)
finetuned_path

'/opt/app-root/src/shared/meta-llama/Llama-3.2-1B-Instruct'

In [22]:
print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(base_model, finetuned_path)
model.config.pad_token_id = model.config.eos_token_id[0]  # 取第一个 EOS token

print("Merging LoRA weights into base model...")
model = model.merge_and_unload()

Loading LoRA adapter...
Merging LoRA weights into base model...


# Save Trained Model

In [23]:
merged_dir = os.path.join(finetuned_path, merged_subdir)

In [24]:
print(f"Saving merged model to {merged_dir}")
os.makedirs(merged_dir, exist_ok=True)
model.save_pretrained(merged_dir)


Saving merged model to /opt/app-root/src/shared/meta-llama/Llama-3.2-1B-Instruct/merged_model


In [25]:
tokenizer.save_pretrained(merged_dir)
print("Saved tokenizer")

Saved tokenizer


## Testing the Fine-Tuned Model

In [26]:
# Load the pre-trained model
my_model = AutoModelForCausalLM.from_pretrained(
    merged_dir,
    local_files_only=True,
    torch_dtype=torch.bfloat16,
).to("cuda")

In [27]:
# Configure the tokenizer
tokenizer = AutoTokenizer.from_pretrained(merged_dir)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
if base_model.config.pad_token_id is None:
    base_model.config.pad_token_id = base_model.config.eos_token_id

In [28]:
# Test the fine-tuned model
pipeline = transformers.pipeline(
    "text-generation",
    model=my_model,
    tokenizer=tokenizer,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map="auto",
)

messages = [
    {
        "role": "user",
        "content": "Janet's ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?",
    }
]

outputs = pipeline(messages, max_new_tokens=256, temperature = 0.01)

output2 = ""
for turn in outputs:
    for item in turn["generated_text"]:
        output2 += f"# {item['role']}\n\n{item['content']}\n\n"

display(Markdown(output2))

Device set to use cuda:0


# user

Janet's ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

# assistant

She eats 3 eggs for breakfast every morning, so she has 16 - 3 = <<16-3=13>>13 eggs left.
She bakes 4 muffins every day, so she has 13 - 4 = <<13-4=9>>9 eggs left.
She sells 9 eggs at the farmers' market daily for $2 per egg, so she makes 9 * 2 = $<<9*2=18>>18 every day.
#### 18



# Cleaning Up

## GPU Memory

If you want to start over and test the pre-trained model again, you can free the GPU / accelerator memory with:

In [29]:
# Unload the model from GPU memory
import gc

del base_model, model

gc.collect()
torch.cuda.empty_cache()

In [30]:
from dotenv import set_key, dotenv_values
from pathlib import Path

params_file = f"{local_base_dir}/output-1.env"
Path(params_file).write_text("")

set_key(params_file, "TUNED_MODEL_LOCATION", merged_dir)
set_key(params_file, "MODEL_NAME", model_name)

!cat {params_file}

TUNED_MODEL_LOCATION='/opt/app-root/src/shared/meta-llama/Llama-3.2-1B-Instruct/merged_model'
MODEL_NAME='meta-llama/Llama-3.2-1B-Instruct'


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
